# Demo of GC-HRMS annotation

Based on asari version 1.17. This is the work in asari.annotate.annotate_gcms_full function, but the notebook shows step-by-step details and allows more customization.

The full process can be run on CLI using `asari annotate --workflow GC`.

SL, 2026-05-15

In [ ]:
# use asari 1.17 or later; install ms_entropy if needed
!pip3 install --upgrade asari-metabolomics

import os
import time
import json
import matplotlib.pyplot as plt
from asari.annotate import *

In [ ]:
# Specify input and output; and parameters
input_feature_table = '/Users/lish/li.proj/IndiPHARM/LookAhead/GCMS/lookahead_gcms_asari13_asari_project_311234834/export/full_Feature_table.tsv'
database_file = '/Users/lish/li.proj/asari_project/GCHRMS/Resources/GCHRMS_Database_251217.msp'
KovatsFile = '/Users/lish/li.proj/asari_project/GCHRMS/KovatsIndex_alkanestandards_cuimc.tsv'

output_dir = 'funride'
project_name_handle = '007'         # something to tag in filenames
ms2_tolerance_in_da=0.005
ri_tolerance=50
score_cutoff_cosine=0.5
score_cutoff_entropy=0.4
corr_cutoff=0.6                     # cutoff of feature correlations, to be considered in a cluster
low_peak_filter_factor=100          # can be None. if used, library peaks lower the basepeak/low_peak_filter_factor are excluded 

In [ ]:
# Create output dir
time_stamp = [str(x) for x in time.localtime()[1:6]]
outdir = output_dir + '_' + ''.join(time_stamp)
os.mkdir(outdir)
print(f"Annotation directory: {outdir}.\n\n")

In [ ]:
# infile in asari format
num_samples, list_features = read_features_from_asari_table(open(input_feature_table).read())
ri_model = read_fit_KovatsIndex_rtime(KovatsFile, sep='\t', frac=0.3)
list_features = append_kovats_index(list_features, ri_model)
list_features.sort(key=lambda x: x['peak_area'], reverse=True)
dict_features = {f['id']: f for f in list_features}

feature_dataframe = pd.read_csv(input_feature_table, sep="\t", index_col=0)
feature_dataframe = feature_dataframe.iloc[:, 10:]

In [ ]:
# load GCHRMS Database in MSP or JSON format
list_lib_entries = reformat_gcms_lib( load_gcms_dbfile(database_file), 
                                    filter_factor=low_peak_filter_factor
                                    )
dict_lib_entries = {e.id: e for e in list_lib_entries}
print(f"Imported {len(list_lib_entries)} compound library entries.")

In [ ]:
# Targeted annotation, searching DB against expt features
matched_results = batch_lib_search_score(
    list_lib_entries, list_features, dict_features, feature_dataframe,
    mz_tolerance_da=ms2_tolerance_in_da, ri_tolerance=ri_tolerance,
    cosine_penalty=1, corr_cutoff=corr_cutoff
)
list_empCpds, feature_anno_list = curate_batch_lib_search_result(
    matched_results, 
    mz_tolerance_da=ms2_tolerance_in_da,
    score_cutoff_cosine=score_cutoff_cosine, 
    score_cutoff_entropy=score_cutoff_entropy,
)
write_tsv_feature_anno(feature_anno_list, dict_features, dict_lib_entries, 
                    os.path.join(outdir, "Features_" + project_name_handle + '.tsv'))
write_tsv_empCpd_anno(list_empCpds, dict_features, dict_lib_entries, 
                    os.path.join(outdir, "empCpds_" + project_name_handle + '.tsv'))
print("Done targeted annotation.")
print(f"Exported tsv results for {len(list_empCpds)} annotated empCpds and {len(set([f['feature'] for f in feature_anno_list]))} unique features.\n")

In [ ]:
# Do single mirror plot, e.g. the 101st list_empCpd
tt = list_empCpds[100]
print(tt['name'] )

_score = f"\n(cosine score: {tt['cosine_score']:.2f}, entropy score: {tt['entropy_score']:.2f})"
_title = tt['id'] + '__' + tt['name']   
mirror_plot(tt['peaks_as_features'], 
            dict_lib_entries[tt['lib_entry_id']].peaks,     # peaks from original lib, not filtered by unit mz match, to show the full spectrum.
            figsize=(12,4),
            match_tol=ms2_tolerance_in_da,
            colors=["blue", "tab:red", "black"],    # colors for top, bottom and peak matching.
            title=_title + _score, 
            # outfile='test.pdf'    # to save pdf
)

In [ ]:
# do all mirror_plots
path_mirrorplots = os.path.join(outdir, "mirrorplots/")
os.makedirs(path_mirrorplots, exist_ok=True)
for tt in list_empCpds:
    # print(tt['name'] )
    _score = f"\n(cosine score: {tt['cosine_score']:.2f}, entropy score: {tt['entropy_score']:.2f})"
    _title = tt['id'] + '__' + tt['name']   
    mirror_plot(tt['peaks_as_features'], 
                # tt['peaks_in_lib'],   # filtered by unit mz match
                dict_lib_entries[tt['lib_entry_id']].peaks,     # peaks from original lib, not filtered by unit mz match, to show the full spectrum.
                match_tol=ms2_tolerance_in_da,
                colors=["blue", "tab:red", "black"],
                title=_title + _score, 
                outfile=os.path.join(path_mirrorplots, sanitize_filename(_title)+'.pdf'))
    plt.close()


In [ ]:
# Export list_empCpds JSON and MSP
list_empCpds = serialize_annotated_empCpds(list_empCpds)
json_output_file = os.path.join(outdir, project_name_handle+'_annotated_pseudospectra.json')
with open(json_output_file, 'w', encoding='utf-8') as O:
    json.dump(list_empCpds, O, indent=2, ensure_ascii=False)
json_pseudospectra_to_msp(list_empCpds, json_output_file.replace('.json', '.msp'))
print(project_name_handle + "_annotated_pseudospectra was written to JSON and MSP formats.\n")

In [ ]:
# De novo construction of pseudospectra (deconvolution). This can take a few minutes.
max_core_features=30000   # where to stop. Not necessary to go to bottom of low-intensity features. 
clustering_step_size = 2000
hcl_distance_cut = 1

core_features = set([f['feature'] for f in feature_anno_list])
list_pseudospectra, core_features = iterative_build_pseudospectra_by_hcl(
    list_features, feature_dataframe, 
    init_core_features=core_features, 
    step_size=clustering_step_size,
    hcl_distance_cut = hcl_distance_cut,
    ri_tolerance=ri_tolerance,
    correlation_cut = corr_cutoff,
    max_core_features=max_core_features 
)

# export results
json_output_file = os.path.join(outdir, project_name_handle+'_denovo_pseudospectra.json')
list_pseudospectra = [port_pseudospectrum_to_json(x, normalize_intensity=True) for x in list_pseudospectra]
with open(json_output_file, 'w', encoding='utf-8') as O:
    json.dump(list_pseudospectra, O, indent=2, ensure_ascii=False)
json_pseudospectra_to_msp(list_pseudospectra, json_output_file.replace('.json', '.msp'))
print(f"Exported {len(list_pseudospectra)} de novo pseudospectra to JSON and MSP formats.\n")


# Summary

One can customize parameters on this notebook. Please note the cells are sequential. 

All pseudospectra are exported to JSON and MSP for further use.